In [9]:
import pickle
import sys
from torch.utils.data import DataLoader
import numpy as np
import torch

sys.path.append("/home/users/dristic/project/Archer")

from multiturn_rl.dataloaders.playpen_dataloader import FlatBufferDataset, custom_collate_fn
from multiturn_rl import dataloaders

sys.modules['dataloaders'] = dataloaders

# Load the training buffer
# buffer_path = "/home/users/dristic/project/Archer/buffers/c2_wordle_critic_v2_buffer/training_buffer.pkl"
buffer_path = "/home/users/dristic/project/Archer/buffers/c2_taboo_v4_buffer/training_buffer.pkl"
with open(buffer_path, "rb") as f:
    buffer = pickle.load(f)



In [10]:
buffer.keys()

dict_keys(['trajectories', 'active_trajectories'])

In [11]:
sumed = 0
counted = 0
for trajectory in buffer['trajectories']:
    episode_score = trajectory[-1]['info']['episode_score']
    if episode_score == 0:
        sumed += len(trajectory)
        counted += 1

print(sumed/counted)

3.0


In [12]:
sumed

258

In [13]:
buffer['trajectories'][2120]

[{'context': [{'role': 'user',
    'content': 'You are playing a collaborative word guessing game in which you have to guess a target word that another player describes to you.\n\nYou can make one guess at each trial. You win when you guess the target word. You lose when you cannot guess it in 3 tries.\n\nAfter each trial you will get a new hint from the other player which starts with CLUE.\n\nMake your guesses by just saying the word using the following form: GUESS: <a word>\n\nLet us start.\n'},
   {'role': 'user',
    'content': 'CLUE: This is a liquid produced by your body, especially when feeling hot or after exercise.'}],
  'response': [{'role': 'assistant', 'content': 'GUESS: Sweat'}],
  'done': True,
  'info': {'response_score': 0,
   'response_feedback': None,
   'episode_score': 100,
   'lost': False,
   'aborted': False,
   'success': True,
   'game_id': 9}}]

In [14]:
trajectories = buffer['trajectories']

In [15]:
steps = []
for trajectory in trajectories:
    for i in range(len(trajectory)):
        current = trajectory[i]
        next_step = trajectory[i + 1] if i + 1 < len(trajectory) else None
        steps.append({
            'obs': current['context'],
            'action': current['response'],
            'reward': torch.tensor(
                current['info']['response_score'] if next_step else current['info']['episode_score'],
                dtype=torch.float
            ),
            'next_obs': next_step['context'] if next_step else current['context'],
            'done': torch.tensor(current['done'], dtype=torch.bool)
        })


In [17]:
steps[122]

{'obs': [{'role': 'user',
   'content': 'You are playing a collaborative word guessing game in which you have to guess a target word that another player describes to you.\n\nYou can make one guess at each trial. You win when you guess the target word. You lose when you cannot guess it in 3 tries.\n\nAfter each trial you will get a new hint from the other player which starts with CLUE.\n\nMake your guesses by just saying the word using the following form: GUESS: <a word>\n\nLet us start.\n'},
  {'role': 'user',
   'content': 'CLUE: This word describes a playful act or statement meant to provoke laughter, but it can also refer to preventing someone from speaking.'},
  {'role': 'assistant', 'content': 'GUESS: Joke'},
  {'role': 'user',
   'content': 'CLUE: While this is often a humorous statement, the word in question can also mean to stop someone from talking or to obstruct their breath.'}],
 'action': [{'role': 'assistant', 'content': 'GUESS: Gag'}],
 'reward': tensor(100.),
 'next_obs'

In [ ]:
print("Buffer loaded successfully!")
print(f"Buffer type: {type(buffer)}")
print(f"Number of trajectories: {len(buffer.trajectories)}")

# Inspect buffer structure
print("\n=== Buffer Structure ===")
if hasattr(buffer, 'trajectories'):
    print(f"Total trajectories: {len(buffer.trajectories)}")
    if buffer.trajectories:
        print(f"First trajectory length: {len(buffer.trajectories[0])}")
        print(f"First step keys: {buffer.trajectories[0][0].keys()}")
        
# Sample some steps
print("\n=== Sampling Steps ===")
sample_steps = buffer.sample_steps()
print(f"Sampled {len(sample_steps)} steps")

# Create dataset
dataset = FlatBufferDataset(sample_steps)
print(f"\n=== Dataset Info ===")
print(f"Dataset size: {len(dataset)}")
print(f"First item keys: {dataset[0].keys()}")

# Analyze rewards
print("\n=== Reward Analysis ===")
rewards = [step['reward'] for step in sample_steps]
rewards_array = np.array(rewards)
print(f"Mean reward: {rewards_array.mean():.4f}")
print(f"Std reward: {rewards_array.std():.4f}")
print(f"Min reward: {rewards_array.min():.4f}")
print(f"Max reward: {rewards_array.max():.4f}")

# Count unique rewards
unique_rewards, counts = np.unique(rewards_array, return_counts=True)
print("\nReward distribution:")
for value, count in zip(unique_rewards, counts):
    print(f"  Reward {value}: {count} occurrences ({count/len(rewards)*100:.1f}%)")

# Analyze game lengths
print("\n=== Game Length Analysis ===")
game_lengths = [len(traj) for traj in buffer.trajectories]
lengths_array = np.array(game_lengths)
print(f"Mean game length: {lengths_array.mean():.2f}")
print(f"Min game length: {lengths_array.min()}")
print(f"Max game length: {lengths_array.max()}")
print(f"Std game length: {lengths_array.std():.2f}")

# Check done flags
print("\n=== Terminal States ===")
done_counts = sum(1 for step in sample_steps if step['done'])
print(f"Terminal steps: {done_counts}/{len(sample_steps)} ({done_counts/len(sample_steps)*100:.1f}%)")

# Inspect a few samples
print("\n=== Sample Observations ===")
for i in range(min(3, len(dataset))):
    print(f"\nSample {i}:")
    print(f"  Obs length: {len(dataset[i]['obs'])}")
    print(f"  Action: {dataset[i]['action'][:100]}...")  # First 100 chars
    print(f"  Reward: {dataset[i]['reward']}")
    print(f"  Done: {dataset[i]['done']}")

In [29]:
dataset[0]

{'obs': [{'role': 'user',
   'content': 'You are a language wizard who likes to guess words by using the given rules.\n\nWelcome to Wordle! You have six attempts to guess the target word, a valid English word of five lowercase letters (a-z). Please use the tags "explanation:" and "guess:" to provide a concise explanation for each guess.\n\nFor instance, if your guess is "apple", your response should be\nexplanation: this is a common five-letter English word, and I am starting my guess with this word.\nguess: apple\n\nAfter each guess, your answer will be validated, and you will receive feedback indicating which letters are correct (green), which letters are correct but in the wrong position (yellow), and which letters are incorrect (red). This feedback can be useful in determining which letters to include or exclude in your next guess.\n\nFor example, the feedback for "apple" might be:\nguess_feedback: a<yellow> p<yellow> p<green> l<yellow> e<red>\n\nThe explanation should contain deta

In [30]:
len(dataset)

256

In [38]:
dataset[1].keys()

dict_keys(['obs', 'action', 'reward', 'next_obs', 'done'])

In [40]:
dataset[0]['reward']

tensor(0.)

In [46]:
len(dataset[50])

5

In [48]:
import pprint

pprint.pprint(dataset[0])



{'action': [{'content': "explanation: since 'e' is in the correct position, "
                        "it's likely a word that contains the letter 'e' and "
                        'might have a common suffix like -se. Given the word '
                        "contains 'u' and all other letters are incorrect, "
                        "I'll try a new guess with a word that includes these "
                        'letters.\n'
                        'guess: used',
             'role': 'assistant'}],
 'done': tensor(True),
 'next_obs': [{'content': 'You are a language wizard who likes to guess words '
                          'by using the given rules.\n'
                          '\n'
                          'Welcome to Wordle! You have six attempts to guess '
                          'the target word, a valid English word of five '
                          'lowercase letters (a-z). Please use the tags '
                          '"explanation:" and "guess:" to provide a concise '

In [51]:
len(dataset[0]['obs'])

3

In [56]:
for i in dataset:
    print(len(i['obs']))

3
9
5
1
5
9
3
5
3
3
1
3
1
7
1
5
5
5
1
3
9
1
9
1
3
1
3
3
1
5
5
1
5
3
5
5
5
7
1
1
3
5
3
1
7
9
5
9
5
7
11
3
11
9
7
5
1
1
9
7
3
7
9
3
1
9
3
9
3
7
11
5
9
3
1
7
7
5
5
3
11
3
5
5
7
3
3
3
5
9
5
3
3
5
3
7
3
7
9
3
1
3
1
3
3
1
7
5
7
1
7
3
7
11
3
3
5
5
1
1
3
3
1
1
9
1
7
5
1
5
5
7
3
7
3
5
5
1
3
11
5
5
1
1
5
9
1
3
1
3
3
5
5
1
11
9
11
3
9
5
3
1
1
9
11
7
5
9
3
9
1
1
3
1
5
1
3
7
7
3
5
5
9
5
3
1
3
3
1
9
1
1
7
1
5
5
1
9
9
3
7
1
11
3
7
7
1
5
11
7
7
1
5
3
7
5
3
7
3
7
1
11
7
3
1
7
7
7
11
5
5
3
1
3
7
9
1
5
3
11
5
5
5
1
7
3
5
5
3
3
9
9
7
5
9
1


In [60]:
dataset[-1]['next_obs']

[{'role': 'user',
  'content': 'You are a language wizard who likes to guess words by using the given rules.\n\nWelcome to Wordle! You have six attempts to guess the target word, a valid English word of five lowercase letters (a-z). Please use the tags "explanation:" and "guess:" to provide a concise explanation for each guess.\n\nFor instance, if your guess is "apple", your response should be\nexplanation: this is a common five-letter English word, and I am starting my guess with this word.\nguess: apple\n\nAfter each guess, your answer will be validated, and you will receive feedback indicating which letters are correct (green), which letters are correct but in the wrong position (yellow), and which letters are incorrect (red). This feedback can be useful in determining which letters to include or exclude in your next guess.\n\nFor example, the feedback for "apple" might be:\nguess_feedback: a<yellow> p<yellow> p<green> l<yellow> e<red>\n\nThe explanation should contain details about

In [87]:
dataset[5]['reward']

tensor(10.)

In [88]:
dataset[5]['action']

[{'role': 'assistant',
  'content': 'explanation: The letter "e" and "t" remain in the correct word, with "e" not in the correct position and "t" in the correct position, and the letters "v" and "n" are incorrect, however "n" is in the word but in a wrong position, my next guess should try "e" in a different position and keep "t" in its correct position, and see if "n" can fit in a different spot.\nguess: benty'}]

In [89]:
dataset[5]['obs']

[{'role': 'user',
  'content': 'You are a language wizard who likes to guess words by using the given rules.\n\nWelcome to Wordle! You have six attempts to guess the target word, a valid English word of five lowercase letters (a-z). Please use the tags "explanation:" and "guess:" to provide a concise explanation for each guess.\n\nFor instance, if your guess is "apple", your response should be\nexplanation: this is a common five-letter English word, and I am starting my guess with this word.\nguess: apple\n\nAfter each guess, your answer will be validated, and you will receive feedback indicating which letters are correct (green), which letters are correct but in the wrong position (yellow), and which letters are incorrect (red). This feedback can be useful in determining which letters to include or exclude in your next guess.\n\nFor example, the feedback for "apple" might be:\nguess_feedback: a<yellow> p<yellow> p<green> l<yellow> e<red>\n\nThe explanation should contain details about

In [90]:
dataset[5]['next_obs']

[{'role': 'user',
  'content': 'You are a language wizard who likes to guess words by using the given rules.\n\nWelcome to Wordle! You have six attempts to guess the target word, a valid English word of five lowercase letters (a-z). Please use the tags "explanation:" and "guess:" to provide a concise explanation for each guess.\n\nFor instance, if your guess is "apple", your response should be\nexplanation: this is a common five-letter English word, and I am starting my guess with this word.\nguess: apple\n\nAfter each guess, your answer will be validated, and you will receive feedback indicating which letters are correct (green), which letters are correct but in the wrong position (yellow), and which letters are incorrect (red). This feedback can be useful in determining which letters to include or exclude in your next guess.\n\nFor example, the feedback for "apple" might be:\nguess_feedback: a<yellow> p<yellow> p<green> l<yellow> e<red>\n\nThe explanation should contain details about

In [81]:
dataset[1]['done']

tensor(True)

In [82]:
dataset[1].keys()

dict_keys(['obs', 'action', 'reward', 'next_obs', 'done'])

In [1]:
import pickle
import sys
from torch.utils.data import DataLoader

sys.path.append("/home/users/dristic/project/Archer")

# from multiturn_rl.dataloaders.playpen_dataloader import FlatBufferDataset, custom_collate_fn
from multiturn_rl import dataloaders
import pickle

sys.modules['dataloaders'] = dataloaders
with open("/home/users/dristic/project/Archer/critic_dataset.pkl", "rb") as f:
    nds = pickle.load(f)

print("Dataset loaded successfully:", nds)

Dataset loaded successfully: <dataloaders.playpen_dataloader.FlatBufferDataset object at 0x72c0b80815a0>


In [2]:
nds[1]['reward']

tensor(-10.)

In [12]:
nds[1]['action'][0]

{'role': 'assistant',
 'content': "explanation: g is now green, and both i and e are also green, which means the word contains these letters. With l being red and d being red, these letters are eliminated. Since i and e are correct, and they're often paired in words, I'll try another word that includes these letters.\nguess: hike"}

In [15]:
for inx, i in enumerate(nds):
    if len(i['action'][0]['content']) < 10:
        print(i['action'])
        print(inx)
        

[{'role': 'assistant', 'content': ''}]
31
[{'role': 'assistant', 'content': ''}]
46
[{'role': 'assistant', 'content': ''}]
108
[{'role': 'assistant', 'content': ''}]
124
[{'role': 'assistant', 'content': ''}]
148
[{'role': 'assistant', 'content': ''}]
166
[{'role': 'assistant', 'content': ''}]
201
[{'role': 'assistant', 'content': ''}]
209
[{'role': 'assistant', 'content': ''}]
222
[{'role': 'assistant', 'content': ''}]
255


In [28]:
nds[255]['action']

[{'role': 'assistant', 'content': ''}]

In [101]:
nds[1]['obs']

[{'role': 'user',
  'content': 'You are a language wizard who likes to guess words by using the given rules.\n\nWelcome to Wordle! You have six attempts to guess the target word, a valid English word of five lowercase letters (a-z). Please use the tags "explanation:" and "guess:" to provide a concise explanation for each guess.\n\nFor instance, if your guess is "apple", your response should be\nexplanation: this is a common five-letter English word, and I am starting my guess with this word.\nguess: apple\n\nAfter each guess, your answer will be validated, and you will receive feedback indicating which letters are correct (green), which letters are correct but in the wrong position (yellow), and which letters are incorrect (red). This feedback can be useful in determining which letters to include or exclude in your next guess.\n\nFor example, the feedback for "apple" might be:\nguess_feedback: a<yellow> p<yellow> p<green> l<yellow> e<red>\n\nThe explanation should contain details about

In [104]:
total = 0
rs = []
for i in nds:
    r = i['reward']
    print(r)
    total +=r
    rs.append(r)
print(f"final: {total/len(nds)}")

tensor(0.)
tensor(-10.)
tensor(10.)
tensor(40.)
tensor(0.)
tensor(40.)
tensor(-10.)
tensor(10.)
tensor(0.)
tensor(100.)
tensor(-30.)
tensor(20.)
tensor(0.)
tensor(100.)
tensor(100.)
tensor(100.)
tensor(100.)
tensor(100.)
tensor(0.)
tensor(-10.)
tensor(30.)
tensor(-10.)
tensor(0.)
tensor(100.)
tensor(0.)
tensor(-10.)
tensor(-10.)
tensor(100.)
tensor(-10.)
tensor(0.)
tensor(-70.)
tensor(-10.)
tensor(-10.)
tensor(100.)
tensor(0.)
tensor(-10.)
tensor(0.)
tensor(60.)
tensor(0.)
tensor(-10.)
tensor(10.)
tensor(-10.)
tensor(100.)
tensor(-50.)
tensor(0.)
tensor(10.)
tensor(-10.)
tensor(100.)
tensor(-10.)
tensor(10.)
tensor(100.)
tensor(0.)
tensor(0.)
tensor(-10.)
tensor(0.)
tensor(-10.)
tensor(-10.)
tensor(100.)
tensor(100.)
tensor(0.)
tensor(100.)
tensor(100.)
tensor(-10.)
tensor(0.)
tensor(10.)
tensor(10.)
tensor(0.)
tensor(100.)
tensor(0.)
tensor(-10.)
tensor(0.)
tensor(0.)
tensor(100.)
tensor(20.)
tensor(100.)
tensor(-10.)
tensor(-20.)
tensor(0.)
tensor(-10.)
tensor(0.)
tensor(-30.)
tensor

In [117]:
nds[10]['action']

[{'role': 'assistant',
  'content': 'explanation: Since "o" is still in the same position but now also "a" is correct, my next guess will be a word that contains only the correct "o" in the same position and the correct "a" in a different position, and the correct "r" in a different position is replaced with another consonant that is often used in English words, which is "t".\nguess: toads'}]

In [118]:
nds[10]

{'obs': [{'role': 'user',
   'content': 'You are a language wizard who likes to guess words by using the given rules.\n\nWelcome to Wordle! You have six attempts to guess the target word, a valid English word of five lowercase letters (a-z). Please use the tags "explanation:" and "guess:" to provide a concise explanation for each guess.\n\nFor instance, if your guess is "apple", your response should be\nexplanation: this is a common five-letter English word, and I am starting my guess with this word.\nguess: apple\n\nAfter each guess, your answer will be validated, and you will receive feedback indicating which letters are correct (green), which letters are correct but in the wrong position (yellow), and which letters are incorrect (red). This feedback can be useful in determining which letters to include or exclude in your next guess.\n\nFor example, the feedback for "apple" might be:\nguess_feedback: a<yellow> p<yellow> p<green> l<yellow> e<red>\n\nThe explanation should contain deta

In [111]:
import numpy as np
pie = np.array(rs)

In [112]:
pie.mean()

18.398438

In [113]:
pie.max()

100.0

0.0

AttributeError: 'numpy.ndarray' object has no attribute 'value_counts'

In [120]:
unique_values, counts = np.unique(pie, return_counts=True)


In [121]:
for value, count in zip(unique_values, counts):
    print(f"Value: {value}, Count: {count}")

Value: -70.0, Count: 1
Value: -50.0, Count: 1
Value: -40.0, Count: 4
Value: -30.0, Count: 6
Value: -20.0, Count: 9
Value: -10.0, Count: 61
Value: 0.0, Count: 86
Value: 10.0, Count: 14
Value: 20.0, Count: 9
Value: 30.0, Count: 6
Value: 40.0, Count: 6
Value: 60.0, Count: 2
Value: 100.0, Count: 51
